# Q1 MuJoCo Playground — Wheel Forward / Backward Train

**Quanta Computer · MIT Taiwan · Q1 Edu SDK**

1. Load **structure reference** from Drive (`1zvOkdKf6-PuOZwCbwdphMXJbkcC-hq13`)
2. Build matching **URDF + MuJoCo MJCF** (CAD dual-wheel feet humanoid)
3. Run **MuJoCo playground** open-loop forward / backward
4. Train **PPO** to track `vx` commands (forward & reverse)

Paste / upload into: https://colab.research.google.com/drive/150OF9mOt5Q_caVPJZgDtuR1g7YIjX9b3?hl=zh-tw

Runtime → GPU optional. Headless GL uses `MUJOCO_GL=egl`.


## 0) Install

In [ ]:
# @title Install packages
%pip -q install "mujoco>=3.1" gymnasium stable-baselines3 gdown imageio imageio-ffmpeg matplotlib pillow tqdm

import os
os.environ["MUJOCO_GL"] = "egl"
print("MUJOCO_GL =", os.environ["MUJOCO_GL"])


## 1) Download structure reference

In [ ]:
# @title Fetch structure PNG from Drive
from pathlib import Path
import gdown
from IPython.display import Image, display

WORK = Path("/content/q1_playground"); WORK.mkdir(parents=True, exist_ok=True)
ASSET = WORK / "assets"; ASSET.mkdir(exist_ok=True)
STRUCT = WORK / "q1_structure_ref.png"

# https://drive.google.com/file/d/1zvOkdKf6-PuOZwCbwdphMXJbkcC-hq13/view
FILE_ID = "1zvOkdKf6-PuOZwCbwdphMXJbkcC-hq13"
gdown.download(id=FILE_ID, output=str(STRUCT), quiet=False)
print(STRUCT, STRUCT.stat().st_size, "bytes")
display(Image(filename=str(STRUCT), width=360))


## 2) Write URDF + MuJoCo MJCF

In [ ]:
# @title Materialize URDF + MJCF (structure-matched)
import base64
from pathlib import Path
import mujoco

WORK = Path("/content/q1_playground")
ASSET = WORK / "assets"; ASSET.mkdir(parents=True, exist_ok=True)
(ASSET / "q1_structure.xml").write_bytes(base64.b64decode("PG11am9jbyBtb2RlbD0icTFfc3RydWN0dXJlX3doZWVsZWQiPgogIDwhLS0gQnVpbHQgZnJvbSBEcml2ZSBDQUQgc3RydWN0dXJlIHJlZiAoZHVhbC13aGVlbCBmb290IGh1bWFub2lkKS4gLS0+CiAgPGNvbXBpbGVyIGFuZ2xlPSJyYWRpYW4iIGF1dG9saW1pdHM9InRydWUiLz4KICA8b3B0aW9uIHRpbWVzdGVwPSIwLjAwMiIgZ3Jhdml0eT0iMCAwIC05LjgxIiBpbnRlZ3JhdG9yPSJpbXBsaWNpdGZhc3QiLz4KCiAgPGRlZmF1bHQ+CiAgICA8am9pbnQgZGFtcGluZz0iMC44IiBhcm1hdHVyZT0iMC4wMiIvPgogICAgPGdlb20gZnJpY3Rpb249IjEuNiAwLjEgMC4wMDUiIGNvbmRpbT0iMyIgc29scmVmPSIwLjAxNSAxIi8+CiAgICA8bW90b3IgY3RybGxpbWl0ZWQ9InRydWUiIGN0cmxyYW5nZT0iLTEgMSIvPgogIDwvZGVmYXVsdD4KCiAgPGFzc2V0PgogICAgPHRleHR1cmUgbmFtZT0iZ3JpZCIgdHlwZT0iMmQiIGJ1aWx0aW49ImNoZWNrZXIiCiAgICAgICAgICAgICByZ2IxPSIwLjIgMC4yNSAwLjMiIHJnYjI9IjAuMTIgMC4xNSAwLjE4IiB3aWR0aD0iNTEyIiBoZWlnaHQ9IjUxMiIvPgogICAgPG1hdGVyaWFsIG5hbWU9ImdyaWQiIHRleHR1cmU9ImdyaWQiIHRleHJlcGVhdD0iOCA4IiByZWZsZWN0YW5jZT0iMC4wNSIvPgogICAgPG1hdGVyaWFsIG5hbWU9ImZyYW1lIiByZ2JhPSIwLjc1IDAuNzYgMC43OCAxIi8+CiAgICA8bWF0ZXJpYWwgbmFtZT0idG9yc29fbWlkIiByZ2JhPSIwLjI1IDAuNTUgMC4zNSAxIi8+CiAgICA8bWF0ZXJpYWwgbmFtZT0icGVsdmlzIiByZ2JhPSIwLjg1IDAuNzUgMC4yNSAxIi8+CiAgICA8bWF0ZXJpYWwgbmFtZT0iaGlwIiByZ2JhPSIwLjU1IDAuNDUgMC43MCAxIi8+CiAgICA8bWF0ZXJpYWwgbmFtZT0iYWN0dWF0b3IiIHJnYmE9IjAuMjUgMC4yNSAwLjI4IDEiLz4KICAgIDxtYXRlcmlhbCBuYW1lPSJ3aGVlbCIgcmdiYT0iMC4xMiAwLjEyIDAuMTQgMSIvPgogICAgPG1hdGVyaWFsIG5hbWU9ImhlYWQiIHJnYmE9IjAuMDggMC4wOCAwLjEwIDEiLz4KICA8L2Fzc2V0PgoKICA8d29ybGRib2R5PgogICAgPGxpZ2h0IHBvcz0iMCAwIDQiIGRpcj0iMCAwIC0xIi8+CiAgICA8Z2VvbSBuYW1lPSJmbG9vciIgdHlwZT0icGxhbmUiIHNpemU9IjI1IDI1IDAuMDUiIG1hdGVyaWFsPSJncmlkIi8+CgogICAgPGJvZHkgbmFtZT0iYmFzZSIgcG9zPSIwIDAgMC43MiI+CiAgICAgIDxmcmVlam9pbnQgbmFtZT0icm9vdCIvPgogICAgICA8aW5lcnRpYWwgcG9zPSIwIDAgMC4wNSIgbWFzcz0iNiIgZGlhZ2luZXJ0aWE9IjAuMDUgMC4wNiAwLjA0Ii8+CiAgICAgIDxnZW9tIHR5cGU9ImJveCIgc2l6ZT0iMC4xMSAwLjA5IDAuMDgiIHBvcz0iMCAwIDAuMDgiIG1hdGVyaWFsPSJwZWx2aXMiLz4KICAgICAgPHNpdGUgbmFtZT0iaW11IiBwb3M9IjAgMCAwLjA4IiBzaXplPSIwLjAxIi8+CgogICAgICA8Ym9keSBuYW1lPSJ0b3Jzb19taWQiIHBvcz0iMCAwIDAuMTYiPgogICAgICAgIDxqb2ludCBuYW1lPSJ3YWlzdF95YXciIHR5cGU9ImhpbmdlIiBheGlzPSIwIDAgMSIgcmFuZ2U9Ii0wLjYgMC42Ii8+CiAgICAgICAgPGluZXJ0aWFsIHBvcz0iMCAwIDAuMDciIG1hc3M9IjMiIGRpYWdpbmVydGlhPSIwLjAyIDAuMDIgMC4wMTUiLz4KICAgICAgICA8Z2VvbSB0eXBlPSJjeWxpbmRlciIgc2l6ZT0iMC4wOSAwLjA3IiBwb3M9IjAgMCAwLjA3IiBtYXRlcmlhbD0idG9yc29fbWlkIi8+CgogICAgICAgIDxib2R5IG5hbWU9InRvcnNvX3VwcGVyIiBwb3M9IjAgMCAwLjE0Ij4KICAgICAgICAgIDxqb2ludCBuYW1lPSJ3YWlzdF9waXRjaCIgdHlwZT0iaGluZ2UiIGF4aXM9IjAgMSAwIiByYW5nZT0iLTAuNCAwLjQiLz4KICAgICAgICAgIDxpbmVydGlhbCBwb3M9IjAgMCAwLjEyIiBtYXNzPSI1IiBkaWFnaW5lcnRpYT0iMC4wNCAwLjA1IDAuMDMiLz4KICAgICAgICAgIDxnZW9tIHR5cGU9ImJveCIgc2l6ZT0iMC4xMiAwLjA5IDAuMTIiIHBvcz0iMCAwIDAuMTIiIG1hdGVyaWFsPSJmcmFtZSIvPgogICAgICAgICAgPGdlb20gdHlwZT0iYm94IiBzaXplPSIwLjA3IDAuMDYgMC4wNiIgcG9zPSIwIDAgMC4zMCIgbWF0ZXJpYWw9ImhlYWQiLz4KCiAgICAgICAgICA8Ym9keSBuYW1lPSJsX2FybSIgcG9zPSIwIDAuMTQgMC4xOCI+CiAgICAgICAgICAgIDxqb2ludCBuYW1lPSJsX3Nob3VsZGVyX3BpdGNoIiB0eXBlPSJoaW5nZSIgYXhpcz0iMCAxIDAiIHJhbmdlPSItMiAxLjUiLz4KICAgICAgICAgICAgPGluZXJ0aWFsIHBvcz0iMCAwIC0wLjEwIiBtYXNzPSIxLjIiIGRpYWdpbmVydGlhPSIwLjAxIDAuMDEgMC4wMDIiLz4KICAgICAgICAgICAgPGdlb20gdHlwZT0iY3lsaW5kZXIiIHNpemU9IjAuMDUgMC4wMyIgcXVhdD0iMC43MDcgMC43MDcgMCAwIiBtYXRlcmlhbD0iYWN0dWF0b3IiLz4KICAgICAgICAgICAgPGdlb20gdHlwZT0iY2Fwc3VsZSIgZnJvbXRvPSIwIDAuMDIgMCAwIDAuMDIgLTAuMjQiIHNpemU9IjAuMDMiIG1hdGVyaWFsPSJmcmFtZSIvPgogICAgICAgICAgICA8Ym9keSBuYW1lPSJsX2ZvcmUiIHBvcz0iMCAwLjAyIC0wLjI0Ij4KICAgICAgICAgICAgICA8am9pbnQgbmFtZT0ibF9lbGJvdyIgdHlwZT0iaGluZ2UiIGF4aXM9IjAgMSAwIiByYW5nZT0iLTIuMiAwIi8+CiAgICAgICAgICAgICAgPGluZXJ0aWFsIHBvcz0iMCAwIC0wLjEwIiBtYXNzPSIwLjgiIGRpYWdpbmVydGlhPSIwLjAwNiAwLjAwNiAwLjAwMSIvPgogICAgICAgICAgICAgIDxnZW9tIHR5cGU9ImNhcHN1bGUiIGZyb210bz0iMCAwIDAgMCAwIC0wLjIwIiBzaXplPSIwLjAyNSIgbWF0ZXJpYWw9ImZyYW1lIi8+CiAgICAgICAgICAgIDwvYm9keT4KICAgICAgICAgIDwvYm9keT4KCiAgICAgICAgICA8Ym9keSBuYW1lPSJyX2FybSIgcG9zPSIwIC0wLjE0IDAuMTgiPgogICAgICAgICAgICA8am9pbnQgbmFtZT0icl9zaG91bGRlcl9waXRjaCIgdHlwZT0iaGluZ2UiIGF4aXM9IjAgMSAwIiByYW5nZT0iLTIgMS41Ii8+CiAgICAgICAgICAgIDxpbmVydGlhbCBwb3M9IjAgMCAtMC4xMCIgbWFzcz0iMS4yIiBkaWFnaW5lcnRpYT0iMC4wMSAwLjAxIDAuMDAyIi8+CiAgICAgICAgICAgIDxnZW9tIHR5cGU9ImN5bGluZGVyIiBzaXplPSIwLjA1IDAuMDMiIHF1YXQ9IjAuNzA3IDAuNzA3IDAgMCIgbWF0ZXJpYWw9ImFjdHVhdG9yIi8+CiAgICAgICAgICAgIDxnZW9tIHR5cGU9ImNhcHN1bGUiIGZyb210bz0iMCAtMC4wMiAwIDAgLTAuMDIgLTAuMjQiIHNpemU9IjAuMDMiIG1hdGVyaWFsPSJmcmFtZSIvPgogICAgICAgICAgICA8Ym9keSBuYW1lPSJyX2ZvcmUiIHBvcz0iMCAtMC4wMiAtMC4yNCI+CiAgICAgICAgICAgICAgPGpvaW50IG5hbWU9InJfZWxib3ciIHR5cGU9ImhpbmdlIiBheGlzPSIwIDEgMCIgcmFuZ2U9Ii0yLjIgMCIvPgogICAgICAgICAgICAgIDxpbmVydGlhbCBwb3M9IjAgMCAtMC4xMCIgbWFzcz0iMC44IiBkaWFnaW5lcnRpYT0iMC4wMDYgMC4wMDYgMC4wMDEiLz4KICAgICAgICAgICAgICA8Z2VvbSB0eXBlPSJjYXBzdWxlIiBmcm9tdG89IjAgMCAwIDAgMCAtMC4yMCIgc2l6ZT0iMC4wMjUiIG1hdGVyaWFsPSJmcmFtZSIvPgogICAgICAgICAgICA8L2JvZHk+CiAgICAgICAgICA8L2JvZHk+CiAgICAgICAgPC9ib2R5PgogICAgICA8L2JvZHk+CgogICAgICA8IS0tIExFRlQgTEVHICsgZHVhbCB3aGVlbHMgLS0+CiAgICAgIDxib2R5IG5hbWU9ImxfdGhpZ2giIHBvcz0iMCAwLjEwIC0wLjAyIj4KICAgICAgICA8am9pbnQgbmFtZT0ibF9oaXBfcGl0Y2giIHR5cGU9ImhpbmdlIiBheGlzPSIwIDEgMCIgcmFuZ2U9Ii0xLjQgMS4wIi8+CiAgICAgICAgPGluZXJ0aWFsIHBvcz0iMCAwIC0wLjE0IiBtYXNzPSIzIiBkaWFnaW5lcnRpYT0iMC4wMyAwLjAzIDAuMDEiLz4KICAgICAgICA8Z2VvbSB0eXBlPSJib3giIHNpemU9IjAuMDQgMC4wMzUgMC4xNCIgcG9zPSIwIDAgLTAuMTQiIG1hdGVyaWFsPSJoaXAiLz4KICAgICAgICA8Ym9keSBuYW1lPSJsX3NoYW5rIiBwb3M9IjAgMCAtMC4yOCI+CiAgICAgICAgICA8am9pbnQgbmFtZT0ibF9rbmVlIiB0eXBlPSJoaW5nZSIgYXhpcz0iMCAxIDAiIHJhbmdlPSIwIDIuMiIvPgogICAgICAgICAgPGluZXJ0aWFsIHBvcz0iMCAwIC0wLjE2IiBtYXNzPSIyLjIiIGRpYWdpbmVydGlhPSIwLjAyNSAwLjAyNSAwLjAwOCIvPgogICAgICAgICAgPGdlb20gdHlwZT0iYm94IiBzaXplPSIwLjAzNSAwLjAzIDAuMTYiIHBvcz0iMCAwIC0wLjE2IiBtYXRlcmlhbD0iZnJhbWUiLz4KICAgICAgICAgIDxib2R5IG5hbWU9Imxfd2hlZWxfaW5uZXIiIHBvcz0iMCAwLjA0NSAtMC4zNCI+CiAgICAgICAgICAgIDxqb2ludCBuYW1lPSJsX3doZWVsX2lubmVyIiB0eXBlPSJoaW5nZSIgYXhpcz0iMCAxIDAiIGRhbXBpbmc9IjAuMDUiLz4KICAgICAgICAgICAgPGluZXJ0aWFsIHBvcz0iMCAwIDAiIG1hc3M9IjAuNiIgZGlhZ2luZXJ0aWE9IjAuMDAyIDAuMDAzIDAuMDAyIi8+CiAgICAgICAgICAgIDxnZW9tIHR5cGU9ImN5bGluZGVyIiBzaXplPSIwLjA5IDAuMDI1IiBtYXRlcmlhbD0id2hlZWwiIGZyaWN0aW9uPSIxLjggMC4xIDAuMDA1Ii8+CiAgICAgICAgICA8L2JvZHk+CiAgICAgICAgICA8Ym9keSBuYW1lPSJsX3doZWVsX291dGVyIiBwb3M9IjAgLTAuMDQ1IC0wLjM0Ij4KICAgICAgICAgICAgPGpvaW50IG5hbWU9Imxfd2hlZWxfb3V0ZXIiIHR5cGU9ImhpbmdlIiBheGlzPSIwIDEgMCIgZGFtcGluZz0iMC4wNSIvPgogICAgICAgICAgICA8aW5lcnRpYWwgcG9zPSIwIDAgMCIgbWFzcz0iMC42IiBkaWFnaW5lcnRpYT0iMC4wMDIgMC4wMDMgMC4wMDIiLz4KICAgICAgICAgICAgPGdlb20gdHlwZT0iY3lsaW5kZXIiIHNpemU9IjAuMDkgMC4wMjUiIG1hdGVyaWFsPSJ3aGVlbCIgZnJpY3Rpb249IjEuOCAwLjEgMC4wMDUiLz4KICAgICAgICAgIDwvYm9keT4KICAgICAgICA8L2JvZHk+CiAgICAgIDwvYm9keT4KCiAgICAgIDwhLS0gUklHSFQgTEVHICsgZHVhbCB3aGVlbHMgLS0+CiAgICAgIDxib2R5IG5hbWU9InJfdGhpZ2giIHBvcz0iMCAtMC4xMCAtMC4wMiI+CiAgICAgICAgPGpvaW50IG5hbWU9InJfaGlwX3BpdGNoIiB0eXBlPSJoaW5nZSIgYXhpcz0iMCAxIDAiIHJhbmdlPSItMS40IDEuMCIvPgogICAgICAgIDxpbmVydGlhbCBwb3M9IjAgMCAtMC4xNCIgbWFzcz0iMyIgZGlhZ2luZXJ0aWE9IjAuMDMgMC4wMyAwLjAxIi8+CiAgICAgICAgPGdlb20gdHlwZT0iYm94IiBzaXplPSIwLjA0IDAuMDM1IDAuMTQiIHBvcz0iMCAwIC0wLjE0IiBtYXRlcmlhbD0iaGlwIi8+CiAgICAgICAgPGJvZHkgbmFtZT0icl9zaGFuayIgcG9zPSIwIDAgLTAuMjgiPgogICAgICAgICAgPGpvaW50IG5hbWU9InJfa25lZSIgdHlwZT0iaGluZ2UiIGF4aXM9IjAgMSAwIiByYW5nZT0iMCAyLjIiLz4KICAgICAgICAgIDxpbmVydGlhbCBwb3M9IjAgMCAtMC4xNiIgbWFzcz0iMi4yIiBkaWFnaW5lcnRpYT0iMC4wMjUgMC4wMjUgMC4wMDgiLz4KICAgICAgICAgIDxnZW9tIHR5cGU9ImJveCIgc2l6ZT0iMC4wMzUgMC4wMyAwLjE2IiBwb3M9IjAgMCAtMC4xNiIgbWF0ZXJpYWw9ImZyYW1lIi8+CiAgICAgICAgICA8Ym9keSBuYW1lPSJyX3doZWVsX2lubmVyIiBwb3M9IjAgLTAuMDQ1IC0wLjM0Ij4KICAgICAgICAgICAgPGpvaW50IG5hbWU9InJfd2hlZWxfaW5uZXIiIHR5cGU9ImhpbmdlIiBheGlzPSIwIDEgMCIgZGFtcGluZz0iMC4wNSIvPgogICAgICAgICAgICA8aW5lcnRpYWwgcG9zPSIwIDAgMCIgbWFzcz0iMC42IiBkaWFnaW5lcnRpYT0iMC4wMDIgMC4wMDMgMC4wMDIiLz4KICAgICAgICAgICAgPGdlb20gdHlwZT0iY3lsaW5kZXIiIHNpemU9IjAuMDkgMC4wMjUiIG1hdGVyaWFsPSJ3aGVlbCIgZnJpY3Rpb249IjEuOCAwLjEgMC4wMDUiLz4KICAgICAgICAgIDwvYm9keT4KICAgICAgICAgIDxib2R5IG5hbWU9InJfd2hlZWxfb3V0ZXIiIHBvcz0iMCAwLjA0NSAtMC4zNCI+CiAgICAgICAgICAgIDxqb2ludCBuYW1lPSJyX3doZWVsX291dGVyIiB0eXBlPSJoaW5nZSIgYXhpcz0iMCAxIDAiIGRhbXBpbmc9IjAuMDUiLz4KICAgICAgICAgICAgPGluZXJ0aWFsIHBvcz0iMCAwIDAiIG1hc3M9IjAuNiIgZGlhZ2luZXJ0aWE9IjAuMDAyIDAuMDAzIDAuMDAyIi8+CiAgICAgICAgICAgIDxnZW9tIHR5cGU9ImN5bGluZGVyIiBzaXplPSIwLjA5IDAuMDI1IiBtYXRlcmlhbD0id2hlZWwiIGZyaWN0aW9uPSIxLjggMC4xIDAuMDA1Ii8+CiAgICAgICAgICA8L2JvZHk+CiAgICAgICAgPC9ib2R5PgogICAgICA8L2JvZHk+CiAgICA8L2JvZHk+CiAgPC93b3JsZGJvZHk+CgogIDxhY3R1YXRvcj4KICAgIDxtb3RvciBuYW1lPSJsX2RyaXZlIiBqb2ludD0ibF93aGVlbF9pbm5lciIgZ2Vhcj0iOCIgY3RybHJhbmdlPSItMSAxIi8+CiAgICA8bW90b3IgbmFtZT0ibF9kcml2ZTIiIGpvaW50PSJsX3doZWVsX291dGVyIiBnZWFyPSI4IiBjdHJscmFuZ2U9Ii0xIDEiLz4KICAgIDxtb3RvciBuYW1lPSJyX2RyaXZlIiBqb2ludD0icl93aGVlbF9pbm5lciIgZ2Vhcj0iOCIgY3RybHJhbmdlPSItMSAxIi8+CiAgICA8bW90b3IgbmFtZT0icl9kcml2ZTIiIGpvaW50PSJyX3doZWVsX291dGVyIiBnZWFyPSI4IiBjdHJscmFuZ2U9Ii0xIDEiLz4KICAgIDxwb3NpdGlvbiBuYW1lPSJsX2hpcCIgam9pbnQ9ImxfaGlwX3BpdGNoIiBrcD0iODAiIGN0cmxyYW5nZT0iLTEuNCAxLjAiLz4KICAgIDxwb3NpdGlvbiBuYW1lPSJyX2hpcCIgam9pbnQ9InJfaGlwX3BpdGNoIiBrcD0iODAiIGN0cmxyYW5nZT0iLTEuNCAxLjAiLz4KICAgIDxwb3NpdGlvbiBuYW1lPSJsX2tuZWUiIGpvaW50PSJsX2tuZWUiIGtwPSIxMDAiIGN0cmxyYW5nZT0iMCAyLjIiLz4KICAgIDxwb3NpdGlvbiBuYW1lPSJyX2tuZWUiIGpvaW50PSJyX2tuZWUiIGtwPSIxMDAiIGN0cmxyYW5nZT0iMCAyLjIiLz4KICAgIDxwb3NpdGlvbiBuYW1lPSJ3YWlzdF95YXdfYWN0IiBqb2ludD0id2Fpc3RfeWF3IiBrcD0iNDAiIGN0cmxyYW5nZT0iLTAuNiAwLjYiLz4KICAgIDxwb3NpdGlvbiBuYW1lPSJ3YWlzdF9waXRjaF9hY3QiIGpvaW50PSJ3YWlzdF9waXRjaCIga3A9IjQwIiBjdHJscmFuZ2U9Ii0wLjQgMC40Ii8+CiAgPC9hY3R1YXRvcj4KCiAgPHNlbnNvcj4KICAgIDxmcmFtZXBvcyBuYW1lPSJiYXNlX3BvcyIgb2JqdHlwZT0iYm9keSIgb2JqbmFtZT0iYmFzZSIvPgogICAgPGZyYW1lbGludmVsIG5hbWU9ImJhc2VfbGludmVsIiBvYmp0eXBlPSJib2R5IiBvYmpuYW1lPSJiYXNlIi8+CiAgPC9zZW5zb3I+CjwvbXVqb2NvPgo="))
(ASSET / "q1_structure.urdf").write_bytes(base64.b64decode("PD94bWwgdmVyc2lvbj0iMS4wIj8+CjwhLS0gUTEgc3RydWN0dXJlIGZyb20gRHJpdmUgQ0FEIHJlZjogd2hlZWxlZCBkdWFsLWZvb3QgaHVtYW5vaWQgLS0+Cjxyb2JvdCBuYW1lPSJxMV9zdHJ1Y3R1cmUiPgogIDxtYXRlcmlhbCBuYW1lPSJmcmFtZSI+PGNvbG9yIHJnYmE9IjAuNzUgMC43NiAwLjc4IDEiLz48L21hdGVyaWFsPgogIDxtYXRlcmlhbCBuYW1lPSJ0b3Jzb19taWQiPjxjb2xvciByZ2JhPSIwLjI1IDAuNTUgMC4zNSAxIi8+PC9tYXRlcmlhbD4KICA8bWF0ZXJpYWwgbmFtZT0icGVsdmlzIj48Y29sb3IgcmdiYT0iMC44NSAwLjc1IDAuMjUgMSIvPjwvbWF0ZXJpYWw+CiAgPG1hdGVyaWFsIG5hbWU9ImhpcCI+PGNvbG9yIHJnYmE9IjAuNTUgMC40NSAwLjcwIDEiLz48L21hdGVyaWFsPgogIDxtYXRlcmlhbCBuYW1lPSJhY3R1YXRvciI+PGNvbG9yIHJnYmE9IjAuMjUgMC4yNSAwLjI4IDEiLz48L21hdGVyaWFsPgogIDxtYXRlcmlhbCBuYW1lPSJ3aGVlbCI+PGNvbG9yIHJnYmE9IjAuMTIgMC4xMiAwLjE0IDEiLz48L21hdGVyaWFsPgogIDxtYXRlcmlhbCBuYW1lPSJoZWFkX2JsayI+PGNvbG9yIHJnYmE9IjAuMDggMC4wOCAwLjEwIDEiLz48L21hdGVyaWFsPgoKICA8bGluayBuYW1lPSJiYXNlX2xpbmsiPgogICAgPHZpc3VhbD48b3JpZ2luIHh5ej0iMCAwIDAuMDgiLz48Z2VvbWV0cnk+PGJveCBzaXplPSIwLjIyIDAuMTggMC4xNiIvPjwvZ2VvbWV0cnk+PG1hdGVyaWFsIG5hbWU9InBlbHZpcyIvPjwvdmlzdWFsPgogICAgPGNvbGxpc2lvbj48b3JpZ2luIHh5ej0iMCAwIDAuMDgiLz48Z2VvbWV0cnk+PGJveCBzaXplPSIwLjIyIDAuMTggMC4xNiIvPjwvZ2VvbWV0cnk+PC9jb2xsaXNpb24+CiAgICA8aW5lcnRpYWw+PG1hc3MgdmFsdWU9IjYuMCIvPjxpbmVydGlhIGl4eD0iMC4wNSIgaXh5PSIwIiBpeHo9IjAiIGl5eT0iMC4wNiIgaXl6PSIwIiBpeno9IjAuMDQiLz48L2luZXJ0aWFsPgogIDwvbGluaz4KCiAgPGxpbmsgbmFtZT0idG9yc29fbWlkIj4KICAgIDx2aXN1YWw+PG9yaWdpbiB4eXo9IjAgMCAwLjA4Ii8+PGdlb21ldHJ5PjxjeWxpbmRlciBsZW5ndGg9IjAuMTQiIHJhZGl1cz0iMC4wOSIvPjwvZ2VvbWV0cnk+PG1hdGVyaWFsIG5hbWU9InRvcnNvX21pZCIvPjwvdmlzdWFsPgogICAgPGNvbGxpc2lvbj48b3JpZ2luIHh5ej0iMCAwIDAuMDgiLz48Z2VvbWV0cnk+PGN5bGluZGVyIGxlbmd0aD0iMC4xNCIgcmFkaXVzPSIwLjA5Ii8+PC9nZW9tZXRyeT48L2NvbGxpc2lvbj4KICAgIDxpbmVydGlhbD48bWFzcyB2YWx1ZT0iMy4wIi8+PGluZXJ0aWEgaXh4PSIwLjAyIiBpeHk9IjAiIGl4ej0iMCIgaXl5PSIwLjAyIiBpeXo9IjAiIGl6ej0iMC4wMTUiLz48L2luZXJ0aWFsPgogIDwvbGluaz4KICA8am9pbnQgbmFtZT0id2Fpc3RfeWF3IiB0eXBlPSJyZXZvbHV0ZSI+CiAgICA8cGFyZW50IGxpbms9ImJhc2VfbGluayIvPjxjaGlsZCBsaW5rPSJ0b3Jzb19taWQiLz4KICAgIDxvcmlnaW4geHl6PSIwIDAgMC4xNiIvPjxheGlzIHh5ej0iMCAwIDEiLz4KICAgIDxsaW1pdCBsb3dlcj0iLTAuNiIgdXBwZXI9IjAuNiIgZWZmb3J0PSI0MCIgdmVsb2NpdHk9IjMiLz4KICA8L2pvaW50PgoKICA8bGluayBuYW1lPSJ0b3Jzb191cHBlciI+CiAgICA8dmlzdWFsPjxvcmlnaW4geHl6PSIwIDAgMC4xMiIvPjxnZW9tZXRyeT48Ym94IHNpemU9IjAuMjQgMC4xOCAwLjI0Ii8+PC9nZW9tZXRyeT48bWF0ZXJpYWwgbmFtZT0iZnJhbWUiLz48L3Zpc3VhbD4KICAgIDxjb2xsaXNpb24+PG9yaWdpbiB4eXo9IjAgMCAwLjEyIi8+PGdlb21ldHJ5Pjxib3ggc2l6ZT0iMC4yNCAwLjE4IDAuMjQiLz48L2dlb21ldHJ5PjwvY29sbGlzaW9uPgogICAgPGluZXJ0aWFsPjxtYXNzIHZhbHVlPSI1LjAiLz48aW5lcnRpYSBpeHg9IjAuMDQiIGl4eT0iMCIgaXh6PSIwIiBpeXk9IjAuMDUiIGl5ej0iMCIgaXp6PSIwLjAzIi8+PC9pbmVydGlhbD4KICA8L2xpbms+CiAgPGpvaW50IG5hbWU9IndhaXN0X3BpdGNoIiB0eXBlPSJyZXZvbHV0ZSI+CiAgICA8cGFyZW50IGxpbms9InRvcnNvX21pZCIvPjxjaGlsZCBsaW5rPSJ0b3Jzb191cHBlciIvPgogICAgPG9yaWdpbiB4eXo9IjAgMCAwLjE0Ii8+PGF4aXMgeHl6PSIwIDEgMCIvPgogICAgPGxpbWl0IGxvd2VyPSItMC40IiB1cHBlcj0iMC40IiBlZmZvcnQ9IjUwIiB2ZWxvY2l0eT0iMyIvPgogIDwvam9pbnQ+CgogIDxsaW5rIG5hbWU9ImhlYWQiPgogICAgPHZpc3VhbD48Z2VvbWV0cnk+PGJveCBzaXplPSIwLjE0IDAuMTIgMC4xMiIvPjwvZ2VvbWV0cnk+PG1hdGVyaWFsIG5hbWU9ImhlYWRfYmxrIi8+PC92aXN1YWw+CiAgICA8aW5lcnRpYWw+PG1hc3MgdmFsdWU9IjEuMCIvPjxpbmVydGlhIGl4eD0iMC4wMDUiIGl4eT0iMCIgaXh6PSIwIiBpeXk9IjAuMDA1IiBpeXo9IjAiIGl6ej0iMC4wMDUiLz48L2luZXJ0aWFsPgogIDwvbGluaz4KICA8am9pbnQgbmFtZT0ibmVjayIgdHlwZT0iZml4ZWQiPgogICAgPHBhcmVudCBsaW5rPSJ0b3Jzb191cHBlciIvPjxjaGlsZCBsaW5rPSJoZWFkIi8+PG9yaWdpbiB4eXo9IjAgMCAwLjMwIi8+CiAgPC9qb2ludD4KCiAgPCEtLSBMRUZUIEFSTSAtLT4KICA8bGluayBuYW1lPSJsX3Nob3VsZGVyIj48dmlzdWFsPjxnZW9tZXRyeT48Y3lsaW5kZXIgbGVuZ3RoPSIwLjA2IiByYWRpdXM9IjAuMDUiLz48L2dlb21ldHJ5PjxtYXRlcmlhbCBuYW1lPSJhY3R1YXRvciIvPjwvdmlzdWFsPgogICAgPGluZXJ0aWFsPjxtYXNzIHZhbHVlPSIwLjgiLz48aW5lcnRpYSBpeHg9IjAuMDAyIiBpeHk9IjAiIGl4ej0iMCIgaXl5PSIwLjAwMiIgaXl6PSIwIiBpeno9IjAuMDAyIi8+PC9pbmVydGlhbD48L2xpbms+CiAgPGpvaW50IG5hbWU9Imxfc2hvdWxkZXJfcGl0Y2giIHR5cGU9InJldm9sdXRlIj4KICAgIDxwYXJlbnQgbGluaz0idG9yc29fdXBwZXIiLz48Y2hpbGQgbGluaz0ibF9zaG91bGRlciIvPgogICAgPG9yaWdpbiB4eXo9IjAgMC4xNCAwLjE4IiBycHk9IjEuNTcwOCAwIDAiLz48YXhpcyB4eXo9IjAgMSAwIi8+CiAgICA8bGltaXQgbG93ZXI9Ii0yLjAiIHVwcGVyPSIxLjUiIGVmZm9ydD0iMzAiIHZlbG9jaXR5PSI0Ii8+CiAgPC9qb2ludD4KICA8bGluayBuYW1lPSJsX3VwcGVyX2FybSI+PHZpc3VhbD48b3JpZ2luIHh5ej0iMCAwIC0wLjEyIi8+PGdlb21ldHJ5PjxjeWxpbmRlciBsZW5ndGg9IjAuMjIiIHJhZGl1cz0iMC4wMyIvPjwvZ2VvbWV0cnk+PG1hdGVyaWFsIG5hbWU9ImZyYW1lIi8+PC92aXN1YWw+CiAgICA8aW5lcnRpYWw+PG1hc3MgdmFsdWU9IjEuMiIvPjxvcmlnaW4geHl6PSIwIDAgLTAuMTIiLz48aW5lcnRpYSBpeHg9IjAuMDEiIGl4eT0iMCIgaXh6PSIwIiBpeXk9IjAuMDEiIGl5ej0iMCIgaXp6PSIwLjAwMiIvPjwvaW5lcnRpYWw+PC9saW5rPgogIDxqb2ludCBuYW1lPSJsX3Nob3VsZGVyX3JvbGwiIHR5cGU9InJldm9sdXRlIj4KICAgIDxwYXJlbnQgbGluaz0ibF9zaG91bGRlciIvPjxjaGlsZCBsaW5rPSJsX3VwcGVyX2FybSIvPgogICAgPG9yaWdpbiB4eXo9IjAgMCAwIi8+PGF4aXMgeHl6PSIxIDAgMCIvPgogICAgPGxpbWl0IGxvd2VyPSItMS4yIiB1cHBlcj0iMS4yIiBlZmZvcnQ9IjI1IiB2ZWxvY2l0eT0iNCIvPgogIDwvam9pbnQ+CiAgPGxpbmsgbmFtZT0ibF9mb3JlYXJtIj48dmlzdWFsPjxvcmlnaW4geHl6PSIwIDAgLTAuMTEiLz48Z2VvbWV0cnk+PGN5bGluZGVyIGxlbmd0aD0iMC4yMCIgcmFkaXVzPSIwLjAyNSIvPjwvZ2VvbWV0cnk+PG1hdGVyaWFsIG5hbWU9ImZyYW1lIi8+PC92aXN1YWw+CiAgICA8aW5lcnRpYWw+PG1hc3MgdmFsdWU9IjAuOCIvPjxvcmlnaW4geHl6PSIwIDAgLTAuMTEiLz48aW5lcnRpYSBpeHg9IjAuMDA2IiBpeHk9IjAiIGl4ej0iMCIgaXl5PSIwLjAwNiIgaXl6PSIwIiBpeno9IjAuMDAxIi8+PC9pbmVydGlhbD48L2xpbms+CiAgPGpvaW50IG5hbWU9ImxfZWxib3ciIHR5cGU9InJldm9sdXRlIj4KICAgIDxwYXJlbnQgbGluaz0ibF91cHBlcl9hcm0iLz48Y2hpbGQgbGluaz0ibF9mb3JlYXJtIi8+CiAgICA8b3JpZ2luIHh5ej0iMCAwIC0wLjI0Ii8+PGF4aXMgeHl6PSIwIDEgMCIvPgogICAgPGxpbWl0IGxvd2VyPSItMi4yIiB1cHBlcj0iMCIgZWZmb3J0PSIyMCIgdmVsb2NpdHk9IjQiLz4KICA8L2pvaW50PgoKICA8IS0tIFJJR0hUIEFSTSAobWlycm9yKSAtLT4KICA8bGluayBuYW1lPSJyX3Nob3VsZGVyIj48dmlzdWFsPjxnZW9tZXRyeT48Y3lsaW5kZXIgbGVuZ3RoPSIwLjA2IiByYWRpdXM9IjAuMDUiLz48L2dlb21ldHJ5PjxtYXRlcmlhbCBuYW1lPSJhY3R1YXRvciIvPjwvdmlzdWFsPgogICAgPGluZXJ0aWFsPjxtYXNzIHZhbHVlPSIwLjgiLz48aW5lcnRpYSBpeHg9IjAuMDAyIiBpeHk9IjAiIGl4ej0iMCIgaXl5PSIwLjAwMiIgaXl6PSIwIiBpeno9IjAuMDAyIi8+PC9pbmVydGlhbD48L2xpbms+CiAgPGpvaW50IG5hbWU9InJfc2hvdWxkZXJfcGl0Y2giIHR5cGU9InJldm9sdXRlIj4KICAgIDxwYXJlbnQgbGluaz0idG9yc29fdXBwZXIiLz48Y2hpbGQgbGluaz0icl9zaG91bGRlciIvPgogICAgPG9yaWdpbiB4eXo9IjAgLTAuMTQgMC4xOCIgcnB5PSItMS41NzA4IDAgMCIvPjxheGlzIHh5ej0iMCAxIDAiLz4KICAgIDxsaW1pdCBsb3dlcj0iLTIuMCIgdXBwZXI9IjEuNSIgZWZmb3J0PSIzMCIgdmVsb2NpdHk9IjQiLz4KICA8L2pvaW50PgogIDxsaW5rIG5hbWU9InJfdXBwZXJfYXJtIj48dmlzdWFsPjxvcmlnaW4geHl6PSIwIDAgLTAuMTIiLz48Z2VvbWV0cnk+PGN5bGluZGVyIGxlbmd0aD0iMC4yMiIgcmFkaXVzPSIwLjAzIi8+PC9nZW9tZXRyeT48bWF0ZXJpYWwgbmFtZT0iZnJhbWUiLz48L3Zpc3VhbD4KICAgIDxpbmVydGlhbD48bWFzcyB2YWx1ZT0iMS4yIi8+PG9yaWdpbiB4eXo9IjAgMCAtMC4xMiIvPjxpbmVydGlhIGl4eD0iMC4wMSIgaXh5PSIwIiBpeHo9IjAiIGl5eT0iMC4wMSIgaXl6PSIwIiBpeno9IjAuMDAyIi8+PC9pbmVydGlhbD48L2xpbms+CiAgPGpvaW50IG5hbWU9InJfc2hvdWxkZXJfcm9sbCIgdHlwZT0icmV2b2x1dGUiPgogICAgPHBhcmVudCBsaW5rPSJyX3Nob3VsZGVyIi8+PGNoaWxkIGxpbms9InJfdXBwZXJfYXJtIi8+CiAgICA8b3JpZ2luIHh5ej0iMCAwIDAiLz48YXhpcyB4eXo9IjEgMCAwIi8+CiAgICA8bGltaXQgbG93ZXI9Ii0xLjIiIHVwcGVyPSIxLjIiIGVmZm9ydD0iMjUiIHZlbG9jaXR5PSI0Ii8+CiAgPC9qb2ludD4KICA8bGluayBuYW1lPSJyX2ZvcmVhcm0iPjx2aXN1YWw+PG9yaWdpbiB4eXo9IjAgMCAtMC4xMSIvPjxnZW9tZXRyeT48Y3lsaW5kZXIgbGVuZ3RoPSIwLjIwIiByYWRpdXM9IjAuMDI1Ii8+PC9nZW9tZXRyeT48bWF0ZXJpYWwgbmFtZT0iZnJhbWUiLz48L3Zpc3VhbD4KICAgIDxpbmVydGlhbD48bWFzcyB2YWx1ZT0iMC44Ii8+PG9yaWdpbiB4eXo9IjAgMCAtMC4xMSIvPjxpbmVydGlhIGl4eD0iMC4wMDYiIGl4eT0iMCIgaXh6PSIwIiBpeXk9IjAuMDA2IiBpeXo9IjAiIGl6ej0iMC4wMDEiLz48L2luZXJ0aWFsPjwvbGluaz4KICA8am9pbnQgbmFtZT0icl9lbGJvdyIgdHlwZT0icmV2b2x1dGUiPgogICAgPHBhcmVudCBsaW5rPSJyX3VwcGVyX2FybSIvPjxjaGlsZCBsaW5rPSJyX2ZvcmVhcm0iLz4KICAgIDxvcmlnaW4geHl6PSIwIDAgLTAuMjQiLz48YXhpcyB4eXo9IjAgMSAwIi8+CiAgICA8bGltaXQgbG93ZXI9Ii0yLjIiIHVwcGVyPSIwIiBlZmZvcnQ9IjIwIiB2ZWxvY2l0eT0iNCIvPgogIDwvam9pbnQ+CgogIDwhLS0gTEVGVCBMRUcgKyBkdWFsIHdoZWVscyAtLT4KICA8bGluayBuYW1lPSJsX2hpcCI+PHZpc3VhbD48Z2VvbWV0cnk+PGN5bGluZGVyIGxlbmd0aD0iMC4wOCIgcmFkaXVzPSIwLjA2Ii8+PC9nZW9tZXRyeT48bWF0ZXJpYWwgbmFtZT0iaGlwIi8+PC92aXN1YWw+CiAgICA8aW5lcnRpYWw+PG1hc3MgdmFsdWU9IjEuNSIvPjxpbmVydGlhIGl4eD0iMC4wMDQiIGl4eT0iMCIgaXh6PSIwIiBpeXk9IjAuMDA0IiBpeXo9IjAiIGl6ej0iMC4wMDQiLz48L2luZXJ0aWFsPjwvbGluaz4KICA8am9pbnQgbmFtZT0ibF9oaXBfeWF3IiB0eXBlPSJyZXZvbHV0ZSI+CiAgICA8cGFyZW50IGxpbms9ImJhc2VfbGluayIvPjxjaGlsZCBsaW5rPSJsX2hpcCIvPgogICAgPG9yaWdpbiB4eXo9IjAgMC4xMCAtMC4wMiIgcnB5PSIxLjU3MDggMCAwIi8+PGF4aXMgeHl6PSIwIDAgMSIvPgogICAgPGxpbWl0IGxvd2VyPSItMC42IiB1cHBlcj0iMC42IiBlZmZvcnQ9IjYwIiB2ZWxvY2l0eT0iNSIvPgogIDwvam9pbnQ+CiAgPGxpbmsgbmFtZT0ibF90aGlnaCI+PHZpc3VhbD48b3JpZ2luIHh5ej0iMCAwIC0wLjE0Ii8+PGdlb21ldHJ5Pjxib3ggc2l6ZT0iMC4wOCAwLjA3IDAuMjgiLz48L2dlb21ldHJ5PjxtYXRlcmlhbCBuYW1lPSJoaXAiLz48L3Zpc3VhbD4KICAgIDxjb2xsaXNpb24+PG9yaWdpbiB4eXo9IjAgMCAtMC4xNCIvPjxnZW9tZXRyeT48Ym94IHNpemU9IjAuMDggMC4wNyAwLjI4Ii8+PC9nZW9tZXRyeT48L2NvbGxpc2lvbj4KICAgIDxpbmVydGlhbD48bWFzcyB2YWx1ZT0iMy4wIi8+PG9yaWdpbiB4eXo9IjAgMCAtMC4xNCIvPjxpbmVydGlhIGl4eD0iMC4wMyIgaXh5PSIwIiBpeHo9IjAiIGl5eT0iMC4wMyIgaXl6PSIwIiBpeno9IjAuMDEiLz48L2luZXJ0aWFsPjwvbGluaz4KICA8am9pbnQgbmFtZT0ibF9oaXBfcGl0Y2giIHR5cGU9InJldm9sdXRlIj4KICAgIDxwYXJlbnQgbGluaz0ibF9oaXAiLz48Y2hpbGQgbGluaz0ibF90aGlnaCIvPgogICAgPG9yaWdpbiB4eXo9IjAgMCAwIiBycHk9Ii0xLjU3MDggMCAwIi8+PGF4aXMgeHl6PSIwIDEgMCIvPgogICAgPGxpbWl0IGxvd2VyPSItMS40IiB1cHBlcj0iMS4wIiBlZmZvcnQ9IjgwIiB2ZWxvY2l0eT0iNiIvPgogIDwvam9pbnQ+CiAgPGxpbmsgbmFtZT0ibF9zaGFuayI+PHZpc3VhbD48b3JpZ2luIHh5ej0iMCAwIC0wLjE2Ii8+PGdlb21ldHJ5Pjxib3ggc2l6ZT0iMC4wNyAwLjA2IDAuMzIiLz48L2dlb21ldHJ5PjxtYXRlcmlhbCBuYW1lPSJmcmFtZSIvPjwvdmlzdWFsPgogICAgPGNvbGxpc2lvbj48b3JpZ2luIHh5ej0iMCAwIC0wLjE2Ii8+PGdlb21ldHJ5Pjxib3ggc2l6ZT0iMC4wNyAwLjA2IDAuMzIiLz48L2dlb21ldHJ5PjwvY29sbGlzaW9uPgogICAgPGluZXJ0aWFsPjxtYXNzIHZhbHVlPSIyLjIiLz48b3JpZ2luIHh5ej0iMCAwIC0wLjE2Ii8+PGluZXJ0aWEgaXh4PSIwLjAyNSIgaXh5PSIwIiBpeHo9IjAiIGl5eT0iMC4wMjUiIGl5ej0iMCIgaXp6PSIwLjAwOCIvPjwvaW5lcnRpYWw+PC9saW5rPgogIDxqb2ludCBuYW1lPSJsX2tuZWUiIHR5cGU9InJldm9sdXRlIj4KICAgIDxwYXJlbnQgbGluaz0ibF90aGlnaCIvPjxjaGlsZCBsaW5rPSJsX3NoYW5rIi8+CiAgICA8b3JpZ2luIHh5ej0iMCAwIC0wLjI4Ii8+PGF4aXMgeHl6PSIwIDEgMCIvPgogICAgPGxpbWl0IGxvd2VyPSIwIiB1cHBlcj0iMi4yIiBlZmZvcnQ9IjEwMCIgdmVsb2NpdHk9IjYiLz4KICA8L2pvaW50PgogIDxsaW5rIG5hbWU9Imxfd2hlZWxfaW5uZXIiPjx2aXN1YWw+PGdlb21ldHJ5PjxjeWxpbmRlciBsZW5ndGg9IjAuMDUiIHJhZGl1cz0iMC4wOSIvPjwvZ2VvbWV0cnk+PG1hdGVyaWFsIG5hbWU9IndoZWVsIi8+PC92aXN1YWw+CiAgICA8Y29sbGlzaW9uPjxnZW9tZXRyeT48Y3lsaW5kZXIgbGVuZ3RoPSIwLjA1IiByYWRpdXM9IjAuMDkiLz48L2dlb21ldHJ5PjwvY29sbGlzaW9uPgogICAgPGluZXJ0aWFsPjxtYXNzIHZhbHVlPSIwLjYiLz48aW5lcnRpYSBpeHg9IjAuMDAyIiBpeHk9IjAiIGl4ej0iMCIgaXl5PSIwLjAwMyIgaXl6PSIwIiBpeno9IjAuMDAyIi8+PC9pbmVydGlhbD48L2xpbms+CiAgPGpvaW50IG5hbWU9Imxfd2hlZWxfaW5uZXIiIHR5cGU9ImNvbnRpbnVvdXMiPgogICAgPHBhcmVudCBsaW5rPSJsX3NoYW5rIi8+PGNoaWxkIGxpbms9Imxfd2hlZWxfaW5uZXIiLz4KICAgIDxvcmlnaW4geHl6PSIwIDAuMDQgLTAuMzQiIHJweT0iMS41NzA4IDAgMCIvPjxheGlzIHh5ej0iMCAwIDEiLz4KICA8L2pvaW50PgogIDxsaW5rIG5hbWU9Imxfd2hlZWxfb3V0ZXIiPjx2aXN1YWw+PGdlb21ldHJ5PjxjeWxpbmRlciBsZW5ndGg9IjAuMDUiIHJhZGl1cz0iMC4wOSIvPjwvZ2VvbWV0cnk+PG1hdGVyaWFsIG5hbWU9IndoZWVsIi8+PC92aXN1YWw+CiAgICA8Y29sbGlzaW9uPjxnZW9tZXRyeT48Y3lsaW5kZXIgbGVuZ3RoPSIwLjA1IiByYWRpdXM9IjAuMDkiLz48L2dlb21ldHJ5PjwvY29sbGlzaW9uPgogICAgPGluZXJ0aWFsPjxtYXNzIHZhbHVlPSIwLjYiLz48aW5lcnRpYSBpeHg9IjAuMDAyIiBpeHk9IjAiIGl4ej0iMCIgaXl5PSIwLjAwMyIgaXl6PSIwIiBpeno9IjAuMDAyIi8+PC9pbmVydGlhbD48L2xpbms+CiAgPGpvaW50IG5hbWU9Imxfd2hlZWxfb3V0ZXIiIHR5cGU9ImNvbnRpbnVvdXMiPgogICAgPHBhcmVudCBsaW5rPSJsX3NoYW5rIi8+PGNoaWxkIGxpbms9Imxfd2hlZWxfb3V0ZXIiLz4KICAgIDxvcmlnaW4geHl6PSIwIC0wLjA0IC0wLjM0IiBycHk9IjEuNTcwOCAwIDAiLz48YXhpcyB4eXo9IjAgMCAxIi8+CiAgPC9qb2ludD4KCiAgPCEtLSBSSUdIVCBMRUcgKyBkdWFsIHdoZWVscyAtLT4KICA8bGluayBuYW1lPSJyX2hpcCI+PHZpc3VhbD48Z2VvbWV0cnk+PGN5bGluZGVyIGxlbmd0aD0iMC4wOCIgcmFkaXVzPSIwLjA2Ii8+PC9nZW9tZXRyeT48bWF0ZXJpYWwgbmFtZT0iaGlwIi8+PC92aXN1YWw+CiAgICA8aW5lcnRpYWw+PG1hc3MgdmFsdWU9IjEuNSIvPjxpbmVydGlhIGl4eD0iMC4wMDQiIGl4eT0iMCIgaXh6PSIwIiBpeXk9IjAuMDA0IiBpeXo9IjAiIGl6ej0iMC4wMDQiLz48L2luZXJ0aWFsPjwvbGluaz4KICA8am9pbnQgbmFtZT0icl9oaXBfeWF3IiB0eXBlPSJyZXZvbHV0ZSI+CiAgICA8cGFyZW50IGxpbms9ImJhc2VfbGluayIvPjxjaGlsZCBsaW5rPSJyX2hpcCIvPgogICAgPG9yaWdpbiB4eXo9IjAgLTAuMTAgLTAuMDIiIHJweT0iLTEuNTcwOCAwIDAiLz48YXhpcyB4eXo9IjAgMCAxIi8+CiAgICA8bGltaXQgbG93ZXI9Ii0wLjYiIHVwcGVyPSIwLjYiIGVmZm9ydD0iNjAiIHZlbG9jaXR5PSI1Ii8+CiAgPC9qb2ludD4KICA8bGluayBuYW1lPSJyX3RoaWdoIj48dmlzdWFsPjxvcmlnaW4geHl6PSIwIDAgLTAuMTQiLz48Z2VvbWV0cnk+PGJveCBzaXplPSIwLjA4IDAuMDcgMC4yOCIvPjwvZ2VvbWV0cnk+PG1hdGVyaWFsIG5hbWU9ImhpcCIvPjwvdmlzdWFsPgogICAgPGNvbGxpc2lvbj48b3JpZ2luIHh5ej0iMCAwIC0wLjE0Ii8+PGdlb21ldHJ5Pjxib3ggc2l6ZT0iMC4wOCAwLjA3IDAuMjgiLz48L2dlb21ldHJ5PjwvY29sbGlzaW9uPgogICAgPGluZXJ0aWFsPjxtYXNzIHZhbHVlPSIzLjAiLz48b3JpZ2luIHh5ej0iMCAwIC0wLjE0Ii8+PGluZXJ0aWEgaXh4PSIwLjAzIiBpeHk9IjAiIGl4ej0iMCIgaXl5PSIwLjAzIiBpeXo9IjAiIGl6ej0iMC4wMSIvPjwvaW5lcnRpYWw+PC9saW5rPgogIDxqb2ludCBuYW1lPSJyX2hpcF9waXRjaCIgdHlwZT0icmV2b2x1dGUiPgogICAgPHBhcmVudCBsaW5rPSJyX2hpcCIvPjxjaGlsZCBsaW5rPSJyX3RoaWdoIi8+CiAgICA8b3JpZ2luIHh5ej0iMCAwIDAiIHJweT0iMS41NzA4IDAgMCIvPjxheGlzIHh5ej0iMCAxIDAiLz4KICAgIDxsaW1pdCBsb3dlcj0iLTEuNCIgdXBwZXI9IjEuMCIgZWZmb3J0PSI4MCIgdmVsb2NpdHk9IjYiLz4KICA8L2pvaW50PgogIDxsaW5rIG5hbWU9InJfc2hhbmsiPjx2aXN1YWw+PG9yaWdpbiB4eXo9IjAgMCAtMC4xNiIvPjxnZW9tZXRyeT48Ym94IHNpemU9IjAuMDcgMC4wNiAwLjMyIi8+PC9nZW9tZXRyeT48bWF0ZXJpYWwgbmFtZT0iZnJhbWUiLz48L3Zpc3VhbD4KICAgIDxjb2xsaXNpb24+PG9yaWdpbiB4eXo9IjAgMCAtMC4xNiIvPjxnZW9tZXRyeT48Ym94IHNpemU9IjAuMDcgMC4wNiAwLjMyIi8+PC9nZW9tZXRyeT48L2NvbGxpc2lvbj4KICAgIDxpbmVydGlhbD48bWFzcyB2YWx1ZT0iMi4yIi8+PG9yaWdpbiB4eXo9IjAgMCAtMC4xNiIvPjxpbmVydGlhIGl4eD0iMC4wMjUiIGl4eT0iMCIgaXh6PSIwIiBpeXk9IjAuMDI1IiBpeXo9IjAiIGl6ej0iMC4wMDgiLz48L2luZXJ0aWFsPjwvbGluaz4KICA8am9pbnQgbmFtZT0icl9rbmVlIiB0eXBlPSJyZXZvbHV0ZSI+CiAgICA8cGFyZW50IGxpbms9InJfdGhpZ2giLz48Y2hpbGQgbGluaz0icl9zaGFuayIvPgogICAgPG9yaWdpbiB4eXo9IjAgMCAtMC4yOCIvPjxheGlzIHh5ej0iMCAxIDAiLz4KICAgIDxsaW1pdCBsb3dlcj0iMCIgdXBwZXI9IjIuMiIgZWZmb3J0PSIxMDAiIHZlbG9jaXR5PSI2Ii8+CiAgPC9qb2ludD4KICA8bGluayBuYW1lPSJyX3doZWVsX2lubmVyIj48dmlzdWFsPjxnZW9tZXRyeT48Y3lsaW5kZXIgbGVuZ3RoPSIwLjA1IiByYWRpdXM9IjAuMDkiLz48L2dlb21ldHJ5PjxtYXRlcmlhbCBuYW1lPSJ3aGVlbCIvPjwvdmlzdWFsPgogICAgPGNvbGxpc2lvbj48Z2VvbWV0cnk+PGN5bGluZGVyIGxlbmd0aD0iMC4wNSIgcmFkaXVzPSIwLjA5Ii8+PC9nZW9tZXRyeT48L2NvbGxpc2lvbj4KICAgIDxpbmVydGlhbD48bWFzcyB2YWx1ZT0iMC42Ii8+PGluZXJ0aWEgaXh4PSIwLjAwMiIgaXh5PSIwIiBpeHo9IjAiIGl5eT0iMC4wMDMiIGl5ej0iMCIgaXp6PSIwLjAwMiIvPjwvaW5lcnRpYWw+PC9saW5rPgogIDxqb2ludCBuYW1lPSJyX3doZWVsX2lubmVyIiB0eXBlPSJjb250aW51b3VzIj4KICAgIDxwYXJlbnQgbGluaz0icl9zaGFuayIvPjxjaGlsZCBsaW5rPSJyX3doZWVsX2lubmVyIi8+CiAgICA8b3JpZ2luIHh5ej0iMCAtMC4wNCAtMC4zNCIgcnB5PSIxLjU3MDggMCAwIi8+PGF4aXMgeHl6PSIwIDAgMSIvPgogIDwvam9pbnQ+CiAgPGxpbmsgbmFtZT0icl93aGVlbF9vdXRlciI+PHZpc3VhbD48Z2VvbWV0cnk+PGN5bGluZGVyIGxlbmd0aD0iMC4wNSIgcmFkaXVzPSIwLjA5Ii8+PC9nZW9tZXRyeT48bWF0ZXJpYWwgbmFtZT0id2hlZWwiLz48L3Zpc3VhbD4KICAgIDxjb2xsaXNpb24+PGdlb21ldHJ5PjxjeWxpbmRlciBsZW5ndGg9IjAuMDUiIHJhZGl1cz0iMC4wOSIvPjwvZ2VvbWV0cnk+PC9jb2xsaXNpb24+CiAgICA8aW5lcnRpYWw+PG1hc3MgdmFsdWU9IjAuNiIvPjxpbmVydGlhIGl4eD0iMC4wMDIiIGl4eT0iMCIgaXh6PSIwIiBpeXk9IjAuMDAzIiBpeXo9IjAiIGl6ej0iMC4wMDIiLz48L2luZXJ0aWFsPjwvbGluaz4KICA8am9pbnQgbmFtZT0icl93aGVlbF9vdXRlciIgdHlwZT0iY29udGludW91cyI+CiAgICA8cGFyZW50IGxpbms9InJfc2hhbmsiLz48Y2hpbGQgbGluaz0icl93aGVlbF9vdXRlciIvPgogICAgPG9yaWdpbiB4eXo9IjAgMC4wNCAtMC4zNCIgcnB5PSIxLjU3MDggMCAwIi8+PGF4aXMgeHl6PSIwIDAgMSIvPgogIDwvam9pbnQ+Cjwvcm9ib3Q+Cg=="))

XML = ASSET / "q1_structure.xml"
model = mujoco.MjModel.from_xml_path(str(XML))
data = mujoco.MjData(model)
mujoco.mj_forward(model, data)
print("URDF:", ASSET / "q1_structure.urdf")
print("MJCF:", XML)
print(f"loaded OK — nq={model.nq} nu={model.nu} nv={model.nv} z={data.qpos[2]:.3f}")


## 3) Playground — open-loop forward / backward

In [ ]:
# @title Record playground MP4
import numpy as np
import mujoco
import imageio.v2 as imageio
from IPython.display import Video, display
from pathlib import Path

WORK = Path("/content/q1_playground")
XML = WORK / "assets" / "q1_structure.xml"
model = mujoco.MjModel.from_xml_path(str(XML))
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model, height=360, width=640)

def set_stance(d):
    d.ctrl[4] = d.ctrl[5] = 0.12
    d.ctrl[6] = d.ctrl[7] = 0.22
    d.ctrl[8] = d.ctrl[9] = 0.0

def set_drive(d, v):
    d.ctrl[0] = d.ctrl[1] = d.ctrl[2] = d.ctrl[3] = float(v)

mujoco.mj_resetData(model, data)
data.qpos[2] = 0.78
set_stance(data)
mujoco.mj_forward(model, data)

frames, fps = [], 30
phases = [(1.5, 0.0), (3.0, 0.45), (1.0, 0.0), (3.0, -0.45), (1.0, 0.0)]
cam = mujoco.MjvCamera(); mujoco.mjv_defaultCamera(cam)
cam.distance, cam.elevation, cam.azimuth = 3.2, -18, 140

for dur, drive in phases:
    for _ in range(int(dur * fps)):
        set_stance(data); set_drive(data, drive)
        for __ in range(max(1, int(1 / (fps * model.opt.timestep)))):
            mujoco.mj_step(model, data)
        cam.lookat[:] = data.qpos[:3]
        renderer.update_scene(data, camera=cam)
        frames.append(renderer.render().copy())

out = WORK / "q1_playground_fb.mp4"
imageio.mimsave(out, frames, fps=fps)
print("wrote", out, "frames", len(frames), "final x=", float(data.qpos[0]), "z=", float(data.qpos[2]))
display(Video(str(out), embed=True, width=640))


## 4) Train PPO — wheel forward / backward

Action: left/right wheel drive. Reward tracks commanded `vx` (+fwd / −back).


In [ ]:
# @title Env
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import mujoco
from pathlib import Path

XML = Path("/content/q1_playground/assets/q1_structure.xml")

class Q1WheelFBEnv(gym.Env):
    metadata = {"render_modes": ["rgb_array"], "render_fps": 50}

    def __init__(self, xml_path=str(XML), episode_steps=300):
        super().__init__()
        self.model = mujoco.MjModel.from_xml_path(xml_path)
        self.data = mujoco.MjData(self.model)
        self.episode_steps = episode_steps
        self.t = 0
        self.cmd_vx = 0.0
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(8,), dtype=np.float32)
        self.action_space = spaces.Box(-1.0, 1.0, shape=(2,), dtype=np.float32)

    def _obs(self):
        d = self.data
        qw, qx, qy, qz = d.qpos[3:7]
        vx, vy, vz = d.qvel[0:3]
        return np.array([self.cmd_vx, vx, vy, vz, d.qpos[2], qx, qy, d.qvel[5]], dtype=np.float32)

    def _stance(self):
        self.data.ctrl[4:8] = [0.12, 0.12, 0.22, 0.22]
        self.data.ctrl[8:10] = 0.0

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        mujoco.mj_resetData(self.model, self.data)
        self.data.qpos[2] = 0.78
        self.data.qpos[3:7] = [1, 0, 0, 0]
        self.cmd_vx = float(self.np_random.choice([-0.55, -0.35, 0.35, 0.55]))
        self.t = 0
        self._stance()
        mujoco.mj_forward(self.model, self.data)
        return self._obs(), {"cmd_vx": self.cmd_vx}

    def step(self, action):
        a = np.clip(np.asarray(action, dtype=np.float64), -1, 1)
        left, right = float(a[0]), float(a[1])
        self._stance()
        self.data.ctrl[0] = self.data.ctrl[1] = left
        self.data.ctrl[2] = self.data.ctrl[3] = right
        for _ in range(10):
            mujoco.mj_step(self.model, self.data)
        self.t += 1
        vx = float(self.data.qvel[0]); z = float(self.data.qpos[2])
        qw, qx, qy, qz = self.data.qpos[3:7]
        track = np.exp(-((vx - self.cmd_vx) ** 2) / 0.0225)
        upright = np.exp(-(qx * qx + qy * qy) / 0.05)
        height = np.exp(-((z - 0.75) ** 2) / 0.01)
        reward = 1.5 * track + 0.5 * upright + 0.3 * height - 0.01 * (left * left + right * right)
        terminated = bool(z < 0.40 or abs(qx) > 0.55 or abs(qy) > 0.55)
        truncated = self.t >= self.episode_steps
        return self._obs(), float(reward), terminated, truncated, {"vx": vx, "cmd_vx": self.cmd_vx}

print("env smoke", Q1WheelFBEnv().reset()[1])


In [ ]:
# @title Train PPO
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
from pathlib import Path

WORK = Path("/content/q1_playground")

def make_env():
    return Monitor(Q1WheelFBEnv())

vec = DummyVecEnv([make_env])
model_rl = PPO(
    "MlpPolicy", vec, verbose=1,
    n_steps=1024, batch_size=256, learning_rate=3e-4,
    gamma=0.99, ent_coef=0.01, device="auto",
)
TIMESTEPS = 80_000  # increase for better tracking
model_rl.learn(total_timesteps=TIMESTEPS, progress_bar=True)
ckpt = WORK / "q1_wheel_fb_ppo.zip"
model_rl.save(str(ckpt))
print("saved", ckpt)


In [ ]:
# @title Evaluate (forward then backward) + MP4
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import mujoco
from IPython.display import Video, display
from pathlib import Path

WORK = Path("/content/q1_playground")
eval_env = Q1WheelFBEnv(episode_steps=250)
xs, vxs, cmds = [], [], []
frames = []
renderer = mujoco.Renderer(eval_env.model, 360, 640)
cam = mujoco.MjvCamera(); mujoco.mjv_defaultCamera(cam)
cam.distance, cam.elevation, cam.azimuth = 3.0, -18, 135

for cmd in (0.45, -0.45):
    obs, _ = eval_env.reset()
    eval_env.cmd_vx = cmd
    obs = eval_env._obs()
    for _ in range(200):
        action, _ = model_rl.predict(obs, deterministic=True)
        obs, r, term, trunc, info = eval_env.step(action)
        xs.append(float(eval_env.data.qpos[0]))
        vxs.append(info["vx"]); cmds.append(cmd)
        cam.lookat[:] = eval_env.data.qpos[:3]
        renderer.update_scene(eval_env.data, camera=cam)
        frames.append(renderer.render().copy())
        if term or trunc:
            break

out = WORK / "q1_trained_fb.mp4"
imageio.mimsave(out, frames, fps=30)
print("wrote", out)

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(cmds, label="cmd"); ax[0].plot(vxs, label="vx"); ax[0].legend(); ax[0].set_title("velocity tracking")
ax[1].plot(xs); ax[1].set_title("x (forward then reverse)")
plt.show()
display(Video(str(out), embed=True, width=640))


## Artifacts

| Item | Path |
|------|------|
| Structure PNG | `/content/q1_playground/q1_structure_ref.png` |
| URDF | `/content/q1_playground/assets/q1_structure.urdf` |
| MuJoCo MJCF | `/content/q1_playground/assets/q1_structure.xml` |
| PPO zip | `/content/q1_playground/q1_wheel_fb_ppo.zip` |
| Playground MP4 | `/content/q1_playground/q1_playground_fb.mp4` |
| Trained MP4 | `/content/q1_playground/q1_trained_fb.mp4` |

**How to update your Colab:** File → Upload notebook → choose `colab/q1_mujoco_playground_wheel_train.ipynb`, or Runtime → Run all after pasting cells into https://colab.research.google.com/drive/150OF9mOt5Q_caVPJZgDtuR1g7YIjX9b3
